# Lab 2A: FMCG Dynamic Pricing & End-to-End Regularized CLV Pipeline
**Course:** Predictive Modelling & Decision Analytics (SoIM MBA)  
**Module 2 (Sessions 3–4)** | **Datasets:** `../datasets/fmcg_dynamic_pricing.csv`, `../datasets/clv_customers.csv`  
*(Note: Time-Series Forecasting—ARIMA, SARIMA, Holt-Winters, Prophet & LSTM—is now in the dedicated companion notebook `Lab_02B_Time_Series_Forecasting_ARIMA_SARIMA_Prophet_LSTM.ipynb`.)*

### Business Case Context
1. **Part A — FMCG Dynamic Pricing:** Estimate Own-Price ($\varepsilon$) and Cross-Price ($\gamma$) elasticities via Log-Log OLS with HC3 robust errors, verify VIF, and compute the Lerner Optimal Price $P^*$.
2. **Part B — E-Commerce CLV Case Study (End-to-End Regularization Lifecycle):**
   - Start from a **Multiple Linear Regression (OLS)** baseline on `clv_customers.csv` with polynomial/interaction features.
   - **Detect Overfitting & Underfitting** quantitatively (Train vs. CV vs. Test $R^2$/RMSE + Learning Curve).
   - Apply **Ridge ($L_2$)**, **Lasso ($L_1$)**, and **ElasticNet ($L_1 + L_2$)** inside `sklearn.pipeline.Pipeline`.
   - **Re-check Overfitting & Underfitting** post-regularization and audit coefficient sparsity.
   - **Deploy & Save** the finalized production pipeline with `joblib` and run a live scoring test.

### Step 1: Import Libraries & Load Datasets
**Why this matters:** Separating imports and verifying shapes early ensures clean reproducibility before any modeling.  
**What the code does:** Imports `pandas`, `numpy`, `statsmodels`, `scikit-learn` (including `PolynomialFeatures`, `RidgeCV`, `LassoCV`, `ElasticNetCV`, `learning_curve`), and `joblib` for model serialization.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

pricing_df = pd.read_csv('../datasets/fmcg_dynamic_pricing.csv')
clv_df = pd.read_csv('../datasets/clv_customers.csv')
print("Pricing shape:", pricing_df.shape, "| CLV shape:", clv_df.shape)
clv_df.head()

**Interpreting Step 1:** `clv_customers.csv` contains 800 customer records with `avg_order_value`, `orders_12m`, `tenure_months`, `return_rate`, `email_open_rate`, `support_calls`, `discount_usage_pct`, plus intentional noise features (`noise_feature_1`, `noise_feature_2`).

---
## Part A (Session 3): FMCG Dynamic Pricing & Elasticity Optimization
### Step 2: Estimate Log-Log Demand Regression & VIF Diagnostics
**Why this matters:** A linear model ($Q = a + bP$) assumes constant unit drops regardless of price level. Log-Log regression ($\ln Q = eta_0 + arepsilon \ln P + \gamma \ln P_{	ext{comp}} + \dots$) directly yields scale-free **percentage elasticities**, while `cov_type='HC3'` protects p-values against heteroskedasticity.  
**What the code does:** Computes natural logs, fits OLS with HC3 robust standard errors, checks VIF (< 5.0), and calculates the Lerner Optimal Price $P^*$.

In [ ]:
pricing_df['ln_q'] = np.log(pricing_df['units_sold'])
pricing_df['ln_p'] = np.log(pricing_df['unit_price'])
pricing_df['ln_comp_p'] = np.log(pricing_df['competitor_price'])

X_p = sm.add_constant(pricing_df[['ln_p', 'ln_comp_p', 'promo_flag', 'store_footfall']])
ols_price = sm.OLS(pricing_df['ln_q'], X_p).fit(cov_type='HC3')
print(ols_price.summary())

eps = ols_price.params['ln_p']
unit_cost = pricing_df['unit_cost'].mean()
p_opt = unit_cost / (1 + 1 / eps)
print(f"\nOwn-Price Elasticity (ε): {eps:.3f} | Cross-Price Elasticity (γ): {ols_price.params['ln_comp_p']:.3f}")
print(f"Average Unit Cost: ${unit_cost:.2f} --> Lerner Optimal Price P*: ${p_opt:.2f}")

vifs = {col: round(variance_inflation_factor(X_p.values, i), 2) for i, col in enumerate(X_p.columns) if col != 'const'}
print("VIF Diagnostics (Healthy if < 5.0):", vifs)

**Interpreting Step 2:**  
- $|arepsilon| > 1$ confirms elastic demand: raising price 1% cuts volume by ~$|arepsilon|\%$.  
- All VIFs $< 5.0$ confirm zero multicollinearity distortion. Category managers can safely use $P^*$ as their profit-maximizing anchor.

---
## Part B (Session 4): E-Commerce CLV — From Multiple Linear Regression to Production Regularization

### Step 3: Train-Test Split & Baseline Multiple Linear Regression (OLS)
**Why this matters:** To detect overfitting and underfitting honestly, we must lock away a **Test Set (25%)** before any scaling or feature expansion. First, we test a **Simple 1-Feature Underfitting Model** vs. a **High-Complexity Polynomial OLS Model (degree=2 interactions)** to observe both failure modes!  
**What the code does:** Splits `clv_customers.csv` (75/25), fits (1) an underfit 1-feature OLS model (`avg_order_value` only) and (2) an unregularized Degree-2 Polynomial OLS model (54 features!), and compares Train vs. Test $R^2$ and RMSE.

In [ ]:
X_c = clv_df.drop(columns=['customer_id', 'clv_24m_usd'])
y_c = clv_df['clv_24m_usd']
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.25, random_state=42)

def eval_fit(name, model, X_train, y_train, X_test, y_test):
    tr_p, te_p = model.predict(X_train), model.predict(X_test)
    tr_r2, te_r2 = r2_score(y_train, tr_p), r2_score(y_test, te_p)
    tr_rmse = root_mean_squared_error(y_train, tr_p)
    te_rmse = root_mean_squared_error(y_test, te_p)
    gap = tr_r2 - te_r2
    print(f"{name:28s} | Train R2: {tr_r2:.3f} | Test R2: {te_r2:.3f} | Gap: {gap:+.3f} | Test RMSE: ${te_rmse:.2f}")

# 1. Underfitting Example (Only 1 feature -> High Bias)
underfit_ols = LinearRegression().fit(X_tr[['avg_order_value']], y_tr)
eval_fit("1. Underfit OLS (1 feature)", underfit_ols, X_tr[['avg_order_value']], y_tr, X_te[['avg_order_value']], y_te)

# 2. Standard Linear OLS (All 9 raw features)
base_ols = Pipeline([('sc', StandardScaler()), ('lr', LinearRegression())]).fit(X_tr, y_tr)
eval_fit("2. Standard MLR (9 features)", base_ols, X_tr, y_tr, X_te, y_te)

# 3. Overfitting Example (Degree-2 Polynomial + Interactions without Regularization -> High Variance)
poly_ols = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('sc', StandardScaler()),
    ('lr', LinearRegression())
]).fit(X_tr, y_tr)
eval_fit("3. Overfit Poly-2 OLS (54 feats)", poly_ols, X_tr, y_tr, X_te, y_te)

**Interpreting Step 3 (Detecting Underfitting vs. Overfitting):**  
- **Underfitting (High Bias):** `1. Underfit OLS` has *both* low Train $R^2$ and low Test $R^2$—it lacks expressive power.  
- **Overfitting (High Variance):** Look at `3. Overfit Poly-2 OLS`—its Train $R^2$ climbs, but its **Train–Test $R^2$ Gap widens** and Test RMSE worsens because 54 unregularized polynomial terms memorize noise!

### Step 4: Visual Overfitting Detection via Learning Curves
**Why this matters:** A **Learning Curve** plots Training vs. Cross-Validation score as sample size grows. A wide persistent gap between the Training curve (top) and Validation curve (bottom) is the textbook signature of **High Variance (Overfitting)**.  
**What the code does:** Plots the 5-fold CV learning curve of the unregularized Poly-2 OLS pipeline.

In [ ]:
train_sizes, tr_scores, val_scores = learning_curve(
    Pipeline([('poly', PolynomialFeatures(degree=2, include_bias=False)), ('sc', StandardScaler()), ('lr', LinearRegression())]),
    X_tr, y_tr, cv=5, scoring='r2', train_sizes=np.linspace(0.2, 1.0, 6)
)
plt.figure(figsize=(8, 4))
plt.plot(train_sizes, tr_scores.mean(axis=1), 'o-', label='Training R2 (Memorization)')
plt.plot(train_sizes, val_scores.mean(axis=1), 'o--', label='5-Fold CV R2 (Generalization)')
plt.title('Overfitting Diagnostic: Unregularized Poly-2 OLS Learning Curve')
plt.xlabel('Training Set Size')
plt.ylabel('R2 Score')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Interpreting Step 4:** The large vertical gap between Training $R^2$ and CV $R^2$ proves the unregularized 54-feature model is overfitting noise. We now apply **Regularization**!

### Step 5: Applying Regularization (Ridge $L_2$, Lasso $L_1$, ElasticNet) & Re-Checking Fit
**Why this matters:** Instead of manually guessing which of the 54 polynomial/interaction terms to keep, we wrap `PolynomialFeatures(2)` + `StandardScaler()` + **CV-tuned Regularization** (`RidgeCV`, `LassoCV`, `ElasticNetCV`) inside a leak-free `Pipeline`.  
**What the code does:** Trains all three regularized pipelines on `X_tr`, prints Train vs. Test $R^2$, the **Overfitting Gap**, and Test RMSE to verify that regularization eliminated overfitting.

In [ ]:
ridge_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('sc', StandardScaler()),
    ('model', RidgeCV(alphas=np.logspace(-1, 4, 40)))
]).fit(X_tr, y_tr)

lasso_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('sc', StandardScaler()),
    ('model', LassoCV(cv=5, max_iter=10000, random_state=42))
]).fit(X_tr, y_tr)

enet_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('sc', StandardScaler()),
    ('model', ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8, 0.95], cv=5, max_iter=10000, random_state=42))
]).fit(X_tr, y_tr)

print("=== Post-Regularization Overfit / Underfit Re-Check ===")
eval_fit("Ridge (L2) Poly-2", ridge_pipe, X_tr, y_tr, X_te, y_te)
eval_fit("Lasso (L1) Poly-2", lasso_pipe, X_tr, y_tr, X_te, y_te)
eval_fit("ElasticNet (L1+L2) Poly-2", enet_pipe, X_tr, y_tr, X_te, y_te)

**Interpreting Step 5 (Re-Check Passed!):**  
Compare the `Gap` and `Test RMSE` against the unregularized Poly-2 OLS from Step 3:
- The **Train–Test $R^2$ Gap shrinks dramatically** (overfitting eliminated!), while **Test $R^2$ improves** over both the underfit 1-feature model and the unregularized 54-feature model!
- Notice that `Lasso (L1)` also captures the true non-linear interaction (`avg_order_value * orders_12m`) while dropping useless noise interactions.

### Step 6: Auditing Sparsity (Which Features Did Lasso Keep?)
**Why this matters:** For C-suite trust, we must inspect which of the 54 polynomial features Lasso kept vs. zeroed out.  
**What the code does:** Extracts feature names from `PolynomialFeatures`, pairs them with `LassoCV` coefficients, and prints the top active drivers and total zeroed-out features.

In [ ]:
feat_names = lasso_pipe.named_steps['poly'].get_feature_names_out(X_c.columns)
coefs = lasso_pipe.named_steps['model'].coef_
coef_df = pd.DataFrame({'Feature': feat_names, 'Lasso_Coef': coefs})
n_zero = (coef_df['Lasso_Coef'] == 0).sum()
print(f"Lasso zeroed out {n_zero} of {len(coef_df)} features! Active drivers:")
print(coef_df[coef_df['Lasso_Coef'] != 0].sort_values('Lasso_Coef', key=abs, ascending=False).head(10).to_string(index=False))

**Interpreting Step 6:** Lasso automatically discovered the exact multiplicative driver `avg_order_value * orders_12m` plus `tenure_months`, `return_rate`, `email_open_rate`, and `support_calls`—while dropping ~40+ noise terms to exact zero!

### Step 7: Production Deployment — Saving & Reloading the Finalized Pipeline
**Why this matters:** In production, you must serialize the **entire `Pipeline`** (polynomial expansion + scaler + regularized model) as a single `.joblib` artifact so live API calls apply identical transformations.  
**What the code does:** Saves `lasso_pipe` to `../models/clv_regularized_pipeline.joblib`, reloads it from disk, and runs a live single-customer CLV prediction.

In [ ]:
os.makedirs('../models', exist_ok=True)
model_path = '../models/clv_regularized_pipeline.joblib'
joblib.dump(lasso_pipe, model_path)
print(f"Saved production CLV pipeline to: {model_path}")

# Simulate Production Inference Service
prod_pipeline = joblib.load(model_path)
new_customer = pd.DataFrame([{
    'avg_order_value': 165.0, 'orders_12m': 14, 'tenure_months': 36,
    'return_rate': 0.05, 'email_open_rate': 0.45, 'support_calls': 1,
    'discount_usage_pct': 0.15, 'noise_feature_1': 0.0, 'noise_feature_2': 0.0
}])
pred_clv = prod_pipeline.predict(new_customer)[0]
print(f"Live API Prediction -> Estimated 24-Month CLV: ${pred_clv:,.2f}")

**Interpreting Step 7:** The reloaded `.joblib` pipeline is 100% self-contained and production-ready for CRM batch scoring or real-time REST APIs.